# viva-biofilm capabilities: spatial structure, performance, and composability

_Investigation `viva-biofilm-capabilities` — coder reproduction notebook._

**Question.** Beyond well-mixed equivalence to the reference model (the viva-biofilm-equivalence
investigation), what does the viva-biofilm Rust + process-bigraph engine
make possible that the Java tool does not: fast, deterministic spatial
biofilm simulation with beautiful interactive visuals, measured performance
scaling, and composability with external control processes?

A demonstration investigation for the viva-biofilm engine's viva-native
capabilities: a spatially developed biofilm (colony structure, solute
gradients, growth curves) rendered as interactive Plotly figures, its
runtime/scaling characteristics, and its composability with a boundary
controller process — all built on the Phase-A Rust core (BiofilmProcess)
now ~23x faster and deterministic after unit reconciliation and
calibration.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-biofilm/viva-biofilm').is_dir():
    REPO = Path('/home/runner/work/viva-biofilm/viva-biofilm')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_biofilm.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

## Study: `spatial-biofilm-growth`

**Question.** Starting from a small inoculum of 40 agents, does the viva-biofilm spatial
engine grow a visibly developed 2D biofilm — hundreds of agents, a
measurable thickness above the substratum, and a genuine substrate-limitation
gradient (boundary richer than substratum) — within a short, deterministic,
non-swept run?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `spatial-biofilm-growth-baseline` | `spatial-biofilm-growth` | 11 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `spatial-biofilm-growth`** — `spec_spatial_biofilm_growth` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_spatial_biofilm_growth = load_spec(REPO / 'viva_biofilm/composites/spatial-biofilm-growth.composite.yaml')
describe_spec(spec_spatial_biofilm_growth)

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: spatial-biofilm-growth ===
STUDY = 'spatial-biofilm-growth'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_spatial_biofilm_growth_baseline = 11
INTERVAL_spatial_biofilm_growth_baseline = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_spatial_biofilm_growth, {'interval': INTERVAL_spatial_biofilm_growth_baseline}, core=core)
        comp.run(STEPS_spatial_biofilm_growth_baseline)  # writes the composite's declared emitter
    print(f'ran 1 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**colony (colored by mass)**


In [ ]:
# colony (colored by mass)
show_viz(_render_one('image:charts/colony_final.html', {'title': 'Final colony', 'caption': 'Agents colored by mass, true-to-scale radius'}, RUNS_DB, STUDY_YAML))

**colony (colored by local substrate)**


In [ ]:
# colony (colored by local substrate)
show_viz(_render_one('image:charts/colony_substrate.html', {'title': 'Final colony — local substrate', 'caption': 'Agents colored by locally sampled substrate concentration'}, RUNS_DB, STUDY_YAML))

**substrate field**


In [ ]:
# substrate field
show_viz(_render_one('image:charts/solute_substrate.html', {'title': 'Substrate field', 'caption': 'Heatmap of the growth-limiting solute at the final step'}, RUNS_DB, STUDY_YAML))

**oxygen field**


In [ ]:
# oxygen field
show_viz(_render_one('image:charts/solute_oxygen.html', {'title': 'Oxygen field', 'caption': 'Heatmap of oxygen at the final step'}, RUNS_DB, STUDY_YAML))

**colony time-lapse**


In [ ]:
# colony time-lapse
show_viz(_render_one('image:charts/timelapse.html', {'title': 'Colony time-lapse', 'caption': 'Animated colony growth across snapshots'}, RUNS_DB, STUDY_YAML))

**growth curves**


In [ ]:
# growth curves
show_viz(_render_one('image:charts/growth_curves.html', {'title': 'Growth curves', 'caption': 'Population, total biomass, and biofilm thickness vs. time'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| developed-biofilm-structure | kind=report_card_axis card=workspace/studies/spatial-biofilm-growth/viz/report_card group=spatial-structure | op verdict_at_least level within_tol |


## Study: `runtime-and-scaling`

**Question.** After the ~23x performance pass and the addition of configurable PDE
parameters, how does the viva-biofilm engine's wall-time scale with grid
size, agent population, and run duration -- and what throughput
(agent-steps/sec) does it sustain?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `runtime-and-scaling-representative` | `runtime-and-scaling` | 11 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `runtime-and-scaling`** — `spec_runtime_and_scaling` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_runtime_and_scaling = load_spec(REPO / 'viva_biofilm/composites/runtime-and-scaling.composite.yaml')
describe_spec(spec_runtime_and_scaling)

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: runtime-and-scaling ===
STUDY = 'runtime-and-scaling'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_runtime_and_scaling_representative = 11
INTERVAL_runtime_and_scaling_representative = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_runtime_and_scaling, {'interval': INTERVAL_runtime_and_scaling_representative}, core=core)
        comp.run(STEPS_runtime_and_scaling_representative)  # writes the composite's declared emitter
    print(f'ran 1 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**grid-size scaling**


In [ ]:
# grid-size scaling
show_viz(_render_one('image:charts/scaling_grid.html', {'title': 'Grid-size scaling', 'caption': 'Wall time per step vs. number of grid cells (log-log), fixed 40 agents / 40 steps'}, RUNS_DB, STUDY_YAML))

**population scaling**


In [ ]:
# population scaling
show_viz(_render_one('image:charts/scaling_population.html', {'title': 'Population scaling', 'caption': 'Wall time per step vs. final agent count (log-log), fixed 48x64 grid / 40 steps'}, RUNS_DB, STUDY_YAML))

**duration scaling**


In [ ]:
# duration scaling
show_viz(_render_one('image:charts/scaling_duration.html', {'title': 'Duration scaling', 'caption': 'Total wall time vs. steps, fixed 48x64 grid / 40 initial agents'}, RUNS_DB, STUDY_YAML))

**throughput**


In [ ]:
# throughput
show_viz(_render_one('image:charts/throughput.html', {'title': 'Throughput', 'caption': 'Agent-steps/sec across the grid and population sweeps'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| throughput-and-scaling-measured | kind=report_card_axis card=workspace/studies/runtime-and-scaling/viz/report_card group=performance | op verdict_at_least level within_tol |


## Study: `composability`

**Question.** Does viva-biofilm's boundary_concentrations runtime hook (the Task 6
set_bulk_by_name binding) actually compose with an external
process-bigraph control process -- can a BoundaryControllerProcess drive
the biofilm's oxygen boundary on a schedule and produce a visible,
measurable growth response, the same way process-bigraph composes any two
processes sharing a store?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `composability-control` | `composability` | 81 | — |
| `composability-perturbed` | `composability` | 81 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `composability`** — `spec_composability` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_composability = load_spec(REPO / 'viva_biofilm/composites/composability.composite.yaml')
describe_spec(spec_composability)

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: composability ===
STUDY = 'composability'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_composability_control = 81
INTERVAL_composability_control = 0.1
STEPS_composability_perturbed = 81
INTERVAL_composability_perturbed = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_composability, {'interval': INTERVAL_composability_control}, core=core)
        comp.run(STEPS_composability_control)  # writes the composite's declared emitter
        comp = build_composite_from_spec(spec_composability, {'interval': INTERVAL_composability_perturbed}, core=core)
        comp.run(STEPS_composability_perturbed)  # writes the composite's declared emitter
    print(f'ran 2 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**response**


In [ ]:
# response
show_viz(_render_one('image:charts/response.html', {'title': 'Biofilm response', 'caption': 'Population, total biomass, and oxygen boundary vs. time — perturbed vs. control, low-oxygen window shaded'}, RUNS_DB, STUDY_YAML))

**colony before perturbation**


In [ ]:
# colony before perturbation
show_viz(_render_one('image:charts/colony_before.html', {'title': 'Colony before perturbation', 'caption': 'Colony state as the oxygen drop begins (t=1d)'}, RUNS_DB, STUDY_YAML))

**colony after perturbation**


In [ ]:
# colony after perturbation
show_viz(_render_one('image:charts/colony_after.html', {'title': 'Colony after perturbation', 'caption': 'Final colony state (t=4d), after the oxygen boundary restored'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| controller-drives-response | kind=report_card_axis card=workspace/studies/composability/viz/report_card group=composability | op verdict_at_least level within_tol |
